In [ ]:
#@markdown

import ipywidgets as widgets
import os, shutil, zipfile, subprocess, glob, re    # Librerías para gestión de archivos, carpetas y procesos del sistema
from pathlib import Path                            # Herramienta para manipular rutas de archivos de forma intuitiva
import pandas as pd                                 # Librería principal para manipulación y análisis de datos (CSVs)
import numpy as np                                  # Librería para operaciones matemáticas y manejo de arreglos numéricos
import cv2                                          # OpenCV: Librería para procesamiento de imágenes y video
import torch                                        # Framework de Deep Learning para ejecutar el modelo YOLO
from google.colab import files                      # Utilidad específica de Colab para subir y descargar archivos
import random                                       # Permite generar números aleatorios para mezclar datos antes del split
import traceback
from IPython.display import display, HTML, clear_output
from google.colab import output
import io
import contextlib
from IPython.display import display, HTML, clear_output
import time
import logging
import math
from skimage.morphology import skeletonize
!pip -q install ultralytics fpdf
from ultralytics import YOLO
from fpdf import FPDF
from datetime import datetime

# --- CONFIGURACIÓN DE RUTAS MAESTRAS ---
BASE = Path("/content")

# Definición de variables globales
DIR_IN = BASE/"in"
DIR_WORK = BASE/"work"
DIR_OUT = BASE/"out"
DIR_DATASET = BASE/"dataset"
DIR_RUNS = BASE/"runs"
DIR_FRAMES = DIR_WORK/"frames"
DIR_RAW = DIR_WORK/"dataset_raw"

# PATHS, para saber que carpetas debe utilizar el entorno
PATHS = {
    "in": DIR_IN,
    "work": DIR_WORK,
    "out": DIR_OUT,
    "frames": DIR_FRAMES,
    "dataset": DIR_DATASET
}

# Creación física de todo el árbol de directorios
for p in [DIR_IN, DIR_WORK, DIR_OUT, DIR_DATASET, DIR_RUNS, DIR_FRAMES, DIR_RAW]:
    p.mkdir(parents=True, exist_ok=True)

# --- CONTENEDOR PRINCIPAL ---
main_container = widgets.VBox()
out = widgets.Output()

STATE = {"paths": PATHS, "out": out}
# ----------------------------------------------------------------------------------------------------------------------------------

def subida_archivos():
    print("📂 Selecciona archivos para añadir")    # Aviso de carga
    subidos = files.upload()                                                   # Selector de archivos

    for nombre_original in subidos.keys():                                     # Itera sobre los archivos subidos
        nombre_min = nombre_original.lower()                                   # Normaliza a minúsculas para comparar
        ruta_destino = PATHS["in"] / nombre_original                           # Conserva nombre original
        print(f"📦 Archivo guardado: {nombre_original}")

        # --- GUARDADO EN DISCO ---
        with open(ruta_destino, "wb") as f:                                    # Abre destino sin borrar el resto de /in
            f.write(subidos[nombre_original])                                  # Escribe el nuevo contenido

def detectar():
    # ==================================================================================================
    # 1. PROCESAMIENTO ESTRATÉGICO: ZIP DE ACTIVOS + VINCULACIÓN DE VIDEO
    # ==================================================================================================

    # 1.1. Extraer el paquete de activos (SRTs, Modelo, CSVs, Logo)
    zip_encontrado = list(PATHS["in"].glob("*.zip"))
    if zip_encontrado:
        print(f"📦 Procesando paquete de activos: {zip_encontrado[0].name}...")
        with zipfile.ZipFile(zip_encontrado[0], 'r') as zip_ref:
            zip_ref.extractall(PATHS["in"])

        # Normalizar ubicación de archivos (sacar de subcarpetas si existen en el ZIP)
        for f in PATHS["in"].rglob("*"):
            if f.is_file():
                destino = PATHS["in"] / f.name
                if f != destino: shutil.move(str(f), str(destino))

    # 1.2. Identificación del Video y Emparejamiento Inteligente de SRT
    video_list = list(PATHS["in"].glob("*.mp4"))
    if not video_list:
        raise SystemExit("❌ ERROR: No se encontró ningún video (.mp4) en la carpeta de entrada.")

    video_path = video_list[0]
    nombre_base_video = video_path.stem  # Ej: "inspeccion_sector_A"
    srt_correspondiente = PATHS["in"] / f"{nombre_base_video}.srt"

    if srt_correspondiente.exists():
        srt_final = srt_correspondiente
        print(f"✅ Telemetría vinculada automáticamente por nombre: {srt_final.name}")
    else:
        srts_disponibles = list(PATHS["in"].glob("*.srt"))
        if srts_disponibles:
            srt_final = srts_disponibles[0]
            print(f"⚠️ Aviso: No hay coincidencia exacta para '{nombre_base_video}.srt'. Usando '{srt_final.name}'")
        else:
            raise SystemExit(f"❌ ERROR: No se encontró ningún archivo SRT para el video {video_path.name}")

    # 1.3. Normalización de Archivos Críticos
    for f in PATHS["in"].glob("*"):
        nombre_min = f.name.lower()
        ext = f.suffix.lower()
        if ext == ".pt":
            f.rename(PATHS["in"] / "best.pt")
        elif "losa" in nombre_min and ext == ".csv":
            f.rename(PATHS["in"] / "losa.csv")
        elif "paramento" in nombre_min and ext == ".csv":
            f.rename(PATHS["in"] / "paramento.csv")
        elif "logo" in nombre_min and ext in [".png", ".jpg", ".jpeg"]:
            f.rename(PATHS["in"] / "logo_gtrh.png")

    print("🚀 Estructura lista. Iniciando procesamiento técnico...")

    # ==================================================================================================
    # 2. PROCESAMIENTO DE TELEMETRÍA Y DATOS TÉCNICOS
    # ==================================================================================================

    def procesar_telemetria_srt(ruta_srt):
        with open(ruta_srt, 'r', encoding='utf-8-sig', errors='ignore') as f:
            contenido = f.read()
        patron_tiempo = r'(\d{1,2}:\d{1,2}:\d{1,2}(?:[.,]\d+)?) --> (\d{1,2}:\d{1,2}:\d{1,2}(?:[.,]\d+)?)'
        tiempos = re.findall(patron_tiempo, contenido)
        textos = re.split(patron_tiempo, contenido)[3::3]
        mapeo = []
        for i, (inicio, fin) in enumerate(tiempos):
            try:
                p = inicio.replace(',', '.').split(':')
                seg = int(p[0]) * 3600 + int(p[1]) * 60 + float(p[2])
                num = re.findall(r"[-+]?\d*\.\d+|\d+", textos[i])
                if num: mapeo.append({'s': seg, 'm': float(num[-1])})
            except: continue
        return pd.DataFrame(mapeo)

    def cargar_datos_robot(ruta):
        try: df = pd.read_csv(ruta, sep=';', decimal=',', encoding='latin-1')
        except: df = pd.read_csv(ruta, sep=';', decimal=',', encoding='cp1252')
        return df.loc[:, ~df.columns.str.contains('^Unnamed')]

    df_telemetria = procesar_telemetria_srt(srt_final)
    df_l = cargar_datos_robot(PATHS["in"] / "losa.csv")
    df_p = cargar_datos_robot(PATHS["in"] / "paramento.csv")
    UMBRAL_DISTANCIA = 2.0

    # ==================================================================================================
    # 3. ANÁLISIS VISUAL (YOLO) Y GEORREFERENCIACIÓN
    # ==================================================================================================

    model = YOLO(PATHS["in"] / "best.pt")
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    duracion_video = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) / fps
    hallazgos_crudos = []
    f_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if f_count % int(fps) == 0:
            seg_act = f_count / fps
            m_act = np.interp(seg_act, df_telemetria['s'], df_telemetria['m'])
            preds = model.predict(frame, conf=0.25, verbose=False)
            for r in preds:
                if len(r.boxes) > 0:
                    frame_plot = r.plot(conf=False, line_width=2, font_size=0.6)
                    h_orig, _, _ = frame.shape
                    y1, y2 = int(h_orig * 0.24), int(h_orig * 0.73)
                    frame_rec = frame_plot[y1:y2, :]
                    img_f = f"evidencia_m_{m_act:.2f}.jpg"
                    cv2.imwrite(str(PATHS["frames"] / img_f), frame_rec)
                    clases = list(set([model.names[int(b.cls)] for b in r.boxes]))
                    hallazgos_crudos.append({
                        'metro': round(m_act, 1), 'seg_video': seg_act,
                        'clases': clases, 'n_clases': len(clases), 'img': img_f
                    })
            print(f"🔍 Escaneando Metro: {m_act:.2f}", end="\r")
        f_count += 1
    cap.release()

    df_vis = pd.DataFrame(hallazgos_crudos).drop_duplicates(subset=['metro'])

    # ==================================================================================================
    # 4. FUSIÓN DE DATOS Y CATEGORIZACIÓN
    # ==================================================================================================

    datos_finales = []
    for _, det in df_vis.iterrows():
        clase_ia = det['clases'][0].lower()
        db, origen = (df_l, "losa.csv") if any(k in clase_ia for k in ["estría", "losa", "fondo"]) else (df_p, "paramento.csv")
        db['diff_lineal'] = (db['Metros'] - det['metro']).abs()
        validos = db[db['diff_lineal'] <= UMBRAL_DISTANCIA]
        res = {**det, 'csv_usado': origen, 'nombre_clase': "Múltiples clases detectadas" if det['n_clases'] > 1 else det['clases'][0].upper()}
        if not validos.empty:
            cercano = validos.nsmallest(1, 'diff_lineal').iloc[0]
            res.update({'tiene_datos': True, 'Magnitud': cercano.get('Magnitud', 'N/A'),
                        'Extensión (cm)': cercano.get('Extensión (cm)', 0), 'Alto (cm)': cercano.get('Alto (cm)', 0),
                        'Área (m2)': cercano.get('Área (m2)', 0), 'Volumen (m3)': cercano.get('Volumen (m3)', 0),
                        'Profundidad (cm)': cercano.get('Profundidad (cm)', 0)})
        else:
            res.update({'tiene_datos': False, 'Profundidad (cm)': -1, 'Magnitud': 'N/A'})
        datos_finales.append(res)

    df_base = pd.DataFrame(datos_finales)
    df_reporte = pd.concat([
        df_base[df_base['n_clases'] >= 3].sort_values('Profundidad (cm)', ascending=False),
        df_base[df_base['n_clases'] == 2].sort_values('Profundidad (cm)', ascending=False),
        df_base[df_base['n_clases'] == 1].sort_values('Profundidad (cm)', ascending=False)
    ])

    # ==================================================================================================
    # 5. GENERACIÓN DE PDF PROFESIONAL (LOGO, KM LIMPIOS, TABLA TÉCNICA COMPLETA)
    # ==================================================================================================

    class PDF(FPDF):
        def header(self):
            logo = PATHS["in"] / "logo_gtrh.png"
            if logo.exists():
                self.image(str(logo), 10, 8, 33)
            self.set_font('Arial', 'B', 10); self.set_text_color(80)
            self.cell(0, 5, 'GERENCIA DE TRANQUES, RELAVES Y RECURSOS HÍDRICOS', ln=True, align='R')
            self.set_font('Arial', '', 8); self.cell(0, 5, 'Codelco, División El Teniente', ln=True, align='R')
            self.ln(10)

        def footer(self):
            self.set_y(-15); self.set_font('Arial', 'I', 8); self.set_text_color(0)
            self.cell(100, 10, 'Informe Técnico de Inspección', 0, 0, 'L')
            self.cell(90, 10, f'Página {self.page_no()}', 0, 0, 'R')

    def fmt_km(val): return f"{int(val):,}".replace(",", ".")
    def fmt_t(s): return f"{int(s//60)}m {int(s%60)}s"

    # Sincronización de páginas
    filas_resumen = len(df_reporte) + 3
    hojas_resumen = (filas_resumen // 25) + 1
    pag_actual = 1 + hojas_resumen + 1
    mapa_paginas = {}
    cat_temp = None
    for idx, f in df_reporte.iterrows():
        c = "S1" if f['n_clases'] >= 3 else ("S2" if f['n_clases'] == 2 else "S3")
        if c != cat_temp: pag_actual += 1; cat_temp = c
        mapa_paginas[idx] = pag_actual; pag_actual += 1

    pdf = PDF()
    pdf.set_auto_page_break(auto=True, margin=20)

    # Portada
    pdf.add_page(); pdf.ln(60); pdf.set_font("Arial", 'B', 22); pdf.cell(190, 15, "INFORME TÉCNICO DE INSPECCIÓN", ln=True, align='C')
    pdf.ln(20); pdf.set_font("Arial", '', 12)
    pdf.cell(190, 8, f"Video analizado: {video_path.name}", ln=True, align='C')
    pdf.cell(190, 8, f"Duración del video: {fmt_t(duracion_video)}", ln=True, align='C')
    pdf.cell(190, 8, f"Modelo computacional utilizado: DAICH - best.pt", ln=True, align='C')
    pdf.cell(190, 8, f"Datos técnicos: losa.csv, paramento.csv", ln=True, align='C')
    pdf.cell(190, 8, f"Fecha de reporte: {datetime.now().strftime('%d/%m/%Y')}", ln=True, align='C')

    # Resumen Técnico
    pdf.add_page(); pdf.set_font("Arial", 'B', 14); pdf.cell(190, 10, "RESUMEN TÉCNICO DE DETECCIONES", ln=True, align='C'); pdf.ln(5)
    w_cols, headers = [20, 25, 85, 20, 25, 10], ["Tiempo", "KM", "Clase de Daño", "Magnitud", "Profundidad (cm)", "Pág"]
    def imprimir_h():
        pdf.set_x(12.5); pdf.set_font("Arial", 'B', 8); pdf.set_fill_color(230)
        for h, w in zip(headers, w_cols): pdf.cell(w, 8, h, 1, 0, 'C', True)
        pdf.ln()

    imprimir_h()
    grupos = [(df_reporte[df_reporte['n_clases'] >= 3], "SECCIÓN I: 3 O MÁS CLASES DETECTADAS"),
              (df_reporte[df_reporte['n_clases'] == 2], "SECCIÓN II: 2 CLASES DETECTADAS"),
              (df_reporte[df_reporte['n_clases'] == 1], "SECCIÓN III: 1 CLASE DETECTADA")]

    for sub, tit in grupos:
        if sub.empty: continue
        pdf.set_x(12.5); pdf.set_fill_color(240); pdf.set_font("Arial", 'B', 8)
        pdf.cell(sum(w_cols), 7, tit, 1, 1, 'C', True); pdf.set_font("Arial", '', 8)
        for idx, r in sub.iterrows():
            if pdf.get_y() > 250: pdf.add_page(); imprimir_h()
            pdf.set_x(12.5)
            pdf.cell(20, 7, fmt_t(r['seg_video']), 1, 0, 'C')
            pdf.cell(25, 7, fmt_km(r['metro']), 1, 0, 'C')
            pdf.cell(85, 7, str(r['nombre_clase'])[:50], 1, 0, 'L')
            pdf.cell(20, 7, str(r['Magnitud']), 1, 0, 'C')
            pdf.cell(25, 7, f"{r['Profundidad (cm)']:.1f}" if r['Profundidad (cm)'] >= 0 else "N/A", 1, 0, 'C')
            pdf.cell(10, 7, str(mapa_paginas[idx]), 1, 1, 'C')

    # Cuerpo de Hallazgos
    cat_act = None
    for idx, f in df_reporte.iterrows():
        nueva_cat = "SECCIÓN I: 3 O MÁS CLASES DETECTADAS" if f['n_clases'] >= 3 else ("SECCIÓN II: 2 CLASES DETECTADAS" if f['n_clases'] == 2 else "SECCIÓN III: 1 CLASE DETECTADA")
        if nueva_cat != cat_act:
            pdf.add_page(); pdf.set_font("Arial", 'B', 16); pdf.ln(80); pdf.cell(190, 20, nueva_cat, ln=True, align='C', border='TB'); cat_act = nueva_cat

        pdf.add_page(); pdf.set_font("Arial", 'B', 12); pdf.cell(0, 10, f"HALLAZGO: {f['nombre_clase']}", ln=True, border='B'); pdf.ln(4)
        pdf.set_font("Arial", 'B', 9); pdf.set_fill_color(245)
        pdf.cell(47.5, 8, " Kilometraje (KM)", 1, 0, 'L', True); pdf.cell(47.5, 8, fmt_km(f['metro']), 1, 0, 'C')
        pdf.cell(47.5, 8, " Tiempo Video", 1, 0, 'L', True); pdf.cell(47.5, 8, fmt_t(f['seg_video']), 1, 1, 'C')

        if f['tiene_datos']:
            pdf.cell(47.5, 8, " Extensión (cm)", 1, 0, 'L', True); pdf.cell(47.5, 8, f"{f['Extensión (cm)']:.2f}", 1, 0, 'C')
            pdf.cell(47.5, 8, " Ancho (cm)", 1, 0, 'L', True); pdf.cell(47.5, 8, f"{f['Alto (cm)']:.2f}", 1, 1, 'C')
            pdf.cell(47.5, 8, " Área (m2)", 1, 0, 'L', True); pdf.cell(47.5, 8, f"{f['Área (m2)']:.4f}", 1, 0, 'C')
            pdf.cell(47.5, 8, " Volumen (m3)", 1, 0, 'L', True); pdf.cell(47.5, 8, f"{f['Volumen (m3)']:.4f}", 1, 1, 'C')
            pdf.cell(47.5, 8, " Profundidad (cm)", 1, 0, 'L', True); pdf.cell(47.5, 8, f"{f['Profundidad (cm)']:.2f}", 1, 0, 'C')
            pdf.cell(47.5, 8, " Magnitud", 1, 0, 'L', True); pdf.cell(47.5, 8, str(f['Magnitud']), 1, 1, 'C')
        else:
            pdf.set_text_color(200, 0, 0); pdf.cell(190, 8, "SIN COINCIDENCIA TÉCNICA EN CSV", 1, 1, 'C'); pdf.set_text_color(0)

        pdf.ln(5); pdf.image(str(PATHS["frames"]/f['img']), x=10, w=190)
        pdf.set_font("Arial", 'I', 7); pdf.cell(190, 5, "Fotograma extraído de videos de inspección registrados por Maquintel", ln=True, align='C')

    # Salida Final
    report_name = f"Reporte_DAICH_{datetime.now().strftime('%Y%m%d_%H%M')}.pdf"
    pdf.output(report_name); files.download(report_name)

def autodimension():
    # ==================================================================================================
    # 1. FUNCIONES TÉCNICAS: TELEMETRÍA Y DIMENSIONAMIENTO (ESQUELETONIZACIÓN)
    # ==================================================================================================

    def calcular_longitud_skeleton(skeleton):
        sk = skeleton.astype("uint8")
        coords = list(zip(*sk.nonzero()))
        length = 0.0
        for y, x in coords:
            for dy in [-1,0,1]:
                for dx in [-1,0,1]:
                    if dx == 0 and dy == 0: continue
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < sk.shape[0] and 0 <= nx < sk.shape[1] and sk[ny, nx]:
                        length += math.sqrt(2) if abs(dx)==1 and abs(dy)==1 else 1
        return length / 2

    def obtener_dimensiones(frame_roi):
        if frame_roi.size == 0: return 0.0, 0.0
        gris = cv2.cvtColor(frame_roi, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        if not np.any(mask): return 0.0, 0.0
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
        if num_labels <= 1: return 0.0, 0.0
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask = (labels == largest).astype(np.uint8) * 255
        dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
        ske = skeletonize(mask > 0)

        px_to_mm = 1000 / float(frame_roi.shape[1])
        l_px = calcular_longitud_skeleton(ske)
        l_cm = (l_px * px_to_mm * 0.169) / 10

        if np.count_nonzero(ske) > 0:
            d_vals = dist[ske > 0] * 2
            w_px = 0.5 * np.median(d_vals) + 0.5 * np.percentile(d_vals, 40)
            a_cm = (w_px * px_to_mm * 0.879) / 10

            # --- Rangos conocidos ---
            if a_cm > 30:
                a_cm -= 10
            elif a_cm < 6.5:
                a_cm += 7.3
        else:
            a_cm = 0.0

        return l_cm, a_cm

    def procesar_telemetria_srt(ruta_srt):
        if not ruta_srt or not os.path.exists(ruta_srt): return pd.DataFrame(columns=['s', 'm'])
        with open(ruta_srt, 'r', encoding='utf-8-sig', errors='ignore') as f:
            contenido = f.read()
        patron_tiempo = r'(\d{1,2}:\d{1,2}:\d{1,2}(?:[.,]\d+)?) --> (\d{1,2}:\d{1,2}:\d{1,2}(?:[.,]\d+)?)'
        tiempos = re.findall(patron_tiempo, contenido)
        textos = re.split(patron_tiempo, contenido)[3::3]
        mapeo = []
        for i, (inicio, fin) in enumerate(tiempos):
            try:
                p = inicio.replace(',', '.').split(':')
                seg = int(p[0]) * 3600 + int(p[1]) * 60 + float(p[2])
                num = re.findall(r"[-+]?\d*\.\d+|\d+", textos[i])
                if num: mapeo.append({'s': seg, 'm': float(num[-1])})
            except: continue
        return pd.DataFrame(mapeo)

    # ==================================================================================================
    # 2. VERIFICACIÓN, EXTRACCIÓN Y MOTOR DE PROCESAMIENTO
    # ==================================================================================================

    zip_engine = list(PATHS["in"].glob("engine.zip")) or list(PATHS["in"].glob("*.zip"))
    video_input = list(PATHS["in"].glob("*.mp4"))

    if not zip_engine or not video_input:
        if not zip_engine: print("❌ ERROR: No se encontró 'engine.zip'")
        if not video_input: print("❌ ERROR: No se encontró video .mp4")
    else:
        with zipfile.ZipFile(zip_engine[0], 'r') as zip_ref:
            zip_ref.extractall(PATHS["in"])

        for f in PATHS["in"].rglob("*"):
            if f.is_file():
                if f.suffix.lower() == ".pt": f.rename(PATHS["in"] / "best.pt")
                elif "logo" in f.name.lower() and f.suffix.lower() in [".png", ".jpg", ".jpeg"]:
                    f.rename(PATHS["in"] / "logo_gtrh.png")
                elif f.parent != PATHS["in"]: shutil.move(str(f), str(PATHS["in"] / f.name))

        video_path = video_input[0]
        srt_list = list(PATHS["in"].glob("*.srt"))
        srt_final = next((s for s in srt_list if s.stem == video_path.stem), srt_list[0] if srt_list else None)

        model = YOLO(PATHS["in"] / "best.pt")
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duracion_total_seg = total_frames / fps
        duracion_str = f"{int(duracion_total_seg // 60)} min {int(duracion_total_seg % 60)} seg"

        df_telemetria = procesar_telemetria_srt(srt_final)
        hallazgos_finales = []
        f_count = 0

        print(f"🚀 Analizando video de {duracion_str}...")

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            if f_count % int(fps) == 0:
                seg_act = f_count / fps
                m_act = np.interp(seg_act, df_telemetria['s'], df_telemetria['m']) if not df_telemetria.empty else 0.0

                preds = model.predict(frame, conf=0.25, verbose=False)
                for r in preds:
                    for i, box_data in enumerate(r.boxes):
                        box = box_data.xyxy[0].cpu().numpy().astype(int)
                        roi = frame[box[1]:box[3], box[0]:box[2]]
                        l_cm, a_cm = obtener_dimensiones(roi)

                        img_f = f"evid_m_{m_act:.2f}_f{f_count}_b{i}.jpg"
                        frame_copy = frame.copy()

                        for other_box in r.boxes:
                            ob = other_box.xyxy[0].cpu().numpy().astype(int)
                            cv2.rectangle(frame_copy, (ob[0], ob[1]), (ob[2], ob[3]), (0, 0, 255), 3)

                        cv2.rectangle(frame_copy, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 4)

                        h_orig = frame.shape[0]
                        frame_rec = frame_copy[int(h_orig * 0.24):int(h_orig * 0.73), :]
                        cv2.imwrite(str(PATHS["frames"] / img_f), frame_rec)

                        hallazgos_finales.append({
                            'metro': round(m_act, 1),
                            'seg_video': seg_act,
                            'clases': model.names[int(box_data.cls)].upper(),
                            'largo': round(l_cm, 1),
                            'ancho': round(a_cm, 1),
                            'img': img_f
                        })
                print(f"🔍 Metro: {m_act:.2f}", end="\r")
            f_count += 1
        cap.release()

        # ==============================================================================================
        # 3. GENERACIÓN DE REPORTE PDF
        # ==============================================================================================

        if hallazgos_finales:
            df_final = pd.DataFrame(hallazgos_finales).sort_values('ancho', ascending=False)

            class PDF(FPDF):
                def header(self):
                    logo = PATHS["in"] / "logo_gtrh.png"
                    if logo.exists(): self.image(str(logo), 10, 8, 33)
                    self.set_font('Arial', 'B', 10); self.set_text_color(0)
                    self.cell(0, 5, 'GERENCIA DE TRANQUES, RELAVES Y RECURSOS HÍDRICOS', ln=True, align='R')
                    self.ln(10)
                def footer(self):
                    self.set_y(-15); self.set_font('Arial', 'I', 8); self.set_text_color(0)
                    self.cell(100, 10, 'Informe técnico de dimensionamiento automatizado', 0, 0, 'L')
                    self.cell(90, 10, f'Página {self.page_no()}', 0, 0, 'R')

            pdf = PDF()
            pdf.set_auto_page_break(auto=True, margin=20)

            pdf.add_page(); pdf.ln(50); pdf.set_font("Arial", 'B', 22); pdf.set_text_color(0)
            pdf.cell(190, 15, "INFORME TÉCNICO DE INSPECCIÓN", ln=True, align='C')
            pdf.ln(10); pdf.set_font("Arial", '', 12)
            pdf.cell(190, 8, f"Video: {video_path.name}", ln=True, align='C')
            pdf.cell(190, 8, f"Duración del video: {duracion_str}", ln=True, align='C')
            pdf.cell(190, 8, f"Fecha: {datetime.now().strftime('%d/%m/%Y')}", ln=True, align='C')
            pdf.cell(190, 8, "Ordenado según: Ancho (mayor a menor)", ln=True, align='C')

            pdf.ln(25); pdf.set_font("Arial", 'B', 11); pdf.set_text_color(200, 0, 0)
            pdf.multi_cell(190, 8, "NOTA: EL DIMENSIONAMIENTO CORRESPONDE A UN MÉTODO AUTOCONTENIDO EN EL ENTORNO. NO UTILIZA DATOS TÉCNICOS DE LA NUBE DE PUNTOS.", align='C')

            pdf.set_text_color(0)
            for _, row in df_final.iterrows():
                pdf.add_page(); pdf.ln(10)
                pdf.set_font("Arial", 'B', 14)
                pdf.cell(0, 10, f"HALLAZGO: {row['clases']}", ln=True, border='B'); pdf.ln(4)

                pdf.set_font("Arial", 'B', 10); pdf.set_fill_color(245)
                pdf.cell(47.5, 9, " Kilometraje (Km)", 1, 0, 'L', True); pdf.cell(47.5, 9, f"{row['metro']:.1f}", 1, 0, 'C')
                pdf.cell(47.5, 9, " Tiempo Video", 1, 0, 'L', True); pdf.cell(47.5, 9, f"{int(row['seg_video']//60)}m {int(row['seg_video']%60)}s", 1, 1, 'C')
                pdf.cell(47.5, 9, " Largo (cm)", 1, 0, 'L', True); pdf.cell(47.5, 9, f"{row['largo']:.1f}", 1, 0, 'C')
                pdf.cell(47.5, 9, " Ancho (cm)", 1, 0, 'L', True); pdf.cell(47.5, 9, f"{row['ancho']:.1f}", 1, 1, 'C')

                pdf.ln(5)
                pdf.image(str(PATHS["frames"] / row['img']), x=10, w=190)
                pdf.set_font("Arial", 'I', 8); pdf.set_text_color(0, 150, 0)
                pdf.cell(190, 6, "(*) Las dimensiones de la tabla corresponden a la falla marcada en VERDE", ln=True, align='C')
                pdf.set_font("Arial", 'I', 7); pdf.set_text_color(0)
                pdf.cell(190, 5, "Fotograma extraído de videos de inspección registrados por Maquintel", ln=True, align='C')

            report_name = f"Reporte_Dimensionado_Ajustado_{datetime.now().strftime('%H%M')}.pdf"
            pdf.output(report_name)
            files.download(report_name)
            print(f"✅ Proceso completo con ajustes de rango. Reporte: {report_name}")

# ==================================================================================================
# 2. UI Y PANEL DE CONTROL (DAICH INTERFACE)
# ==================================================================================================

VIDEO_EXTS = [".mp4", ".mov", ".avi", ".mkv"]

def _find_first_video():
    p_in = PATHS["in"]
    for ext in VIDEO_EXTS:
        vids = list(p_in.glob(f"*{ext}"))
        if vids: return vids[0]
    return None

def _presence():
    p_in = PATHS["in"]
    engine_zip = p_in / "engine.zip"
    has_csv = (p_in/"losa.csv").exists() or (p_in/"paramento.csv").exists() or len(list(p_in.glob("*.csv"))) > 0

    return {
        "best": (p_in/"best.pt") if (p_in/"best.pt").exists() else None,
        "video": _find_first_video(),
        "zip_engine": engine_zip if engine_zip.exists() else None,
        "srt": next(p_in.glob("*.srt"), None),
        "losa": (p_in/"losa.csv") if (p_in/"losa.csv").exists() else None,
        "paramento": (p_in/"paramento.csv") if (p_in/"paramento.csv").exists() else None,
        "has_csv": has_csv,
    }

def _missing(required_keys):
    pres = _presence()
    return [k for k in required_keys if pres.get(k) is None]

def _status_dict():
    pres = _presence()
    f_count = len(list(PATHS["frames"].glob("*.jpg"))) + len(list(PATHS["frames"].glob("*.png")))
    return {
        "zip": pres["zip_engine"].name if pres["zip_engine"] else None,
        "video": pres["video"].name if pres["video"] else None,
        "best": pres["best"].name if pres["best"] else None,
        "frames_count": f_count,
        "yaml_ok": Path("/content/data.yaml").exists()
    }

LABELS = {
    "zip_engine": "Paquete de Activos (engine.zip)",
    "video": "Video de Inspección (.mp4)",
    "best": "Modelo (best.pt)"
}

def _missing_text(required_keys):
    miss = _missing(required_keys)
    return "Ninguno (todo OK)." if not miss else ", ".join([LABELS[k] for k in miss])

# --------------------------------------------------------------------------------------------------
# Estado de la UI
# --------------------------------------------------------------------------------------------------
_UI = {"screen": "welcome", "nav": ["welcome"], "log": ""}

class _SilenceUltralytics:
    def __enter__(self):
        self.saved = []
        for name in ["ultralytics", "ultralytics.yolo", "yolv8", ""]:
            lg = logging.getLogger(name)
            self.saved.append((lg, lg.level, lg.disabled))
            lg.setLevel(logging.ERROR)
        return self
    def __exit__(self, exc_type, exc, tb):
        for lg, level, disabled in self.saved:
            lg.setLevel(level); lg.disabled = disabled

# --------------------------------------------------------------------------------------------------
# Renderizado HTML y CSS
# --------------------------------------------------------------------------------------------------
def _requirements_html(required_keys, title):
    pres = _presence()
    rows = []
    for k in required_keys:
        ok = pres.get(k) is not None
        icon, color, fname = ("✅", "#16a34a", pres[k].name) if ok else ("❌", "#dc2626", "FALTANTE")
        rows.append(f'<div class="reqrow"><span style="color:{color};font-weight:900;">{icon}</span><span class="reqlabel">{LABELS[k]}</span><span class="reqfile"><b>{fname}</b></span></div>')

    missing = _missing(required_keys)
    head, head_color = ("✅ Requisitos completos", "#86efac") if not missing else ("❌ Faltan requisitos", "#fecaca")
    return f'<div class="card"><div class="cardtitle">{title}</div><div class="subbox"><div style="font-weight:900;color:{head_color};margin-bottom:8px;">{head}</div>{"".join(rows)}</div></div>'

def _chip(ok, label, detail):
    color, bg = ("#16a34a", "#062a12") if ok else ("#dc2626", "#2a0a0a")
    return f'<div class="chip" style="border-color:{color}; background:{bg};"><div class="chipttl">{"✅" if ok else "❌"} {label}</div><div class="chipdet">{detail or "no disponible"}</div></div>'

def _status_html():
    s = _status_dict()
    return f"""
    <div class="card">
        <div class="cardtitle">Estado del sistema</div>
        <div class="grid4">
            {_chip(bool(s["zip"]), "engine.zip", s["zip"])}
            {_chip(bool(s["video"]), "Video MP4", s["video"])}
            {_chip(bool(s["best"]), "Modelo PT", s["best"])}
            {_chip(s["frames_count"]>0, "Frames", f'{s["frames_count"]} imágenes')}
        </div>
    </div>
    """

def _btn(label, action, style="primary", disabled=False):
    dis = "disabled" if disabled else ""
    return f"""<button class="btn {style}" {dis} onclick="daichAction('{action}')">{label}</button>"""

def _topbar(title):
    can_back = len(_UI["nav"]) > 1
    return f'<div class="topbar"><div style="width:160px">{_btn("⬅️ Atrás", "back", "ghost", disabled=(not can_back))}</div><div class="toptitle">{title}</div></div>'

def _wrap(inner):
    return f"""
    <style>
        .wrap {{ font-family: ui-sans-serif, system-ui, Arial; color: #e5e7eb; }}
        .card {{ background:#0b0f19; border:1px solid #1f2a44; border-radius:18px; padding:16px 18px; margin-bottom:12px; }}
        .bigtitle {{ font-size:20px; font-weight:1000; }}
        .subtitle {{ color:#94a3b8; margin-top:6px; font-size:13px; line-height:1.35; }}
        .cardtitle {{ font-size:14px; font-weight:1000; margin-bottom:10px; color:#e5e7eb; }}
        .p {{ color:#cbd5e1; font-size:13px; line-height:1.55; }}
        .center {{ display:flex; justify-content:center; }}
        .grid1 {{ display:grid; grid-template-columns: 1fr; gap:10px; }}
        .btn {{ width:100%; padding:14px; border-radius:14px; border:1px solid #2b3a5a; background:#111827; color:#e5e7eb; cursor:pointer; font-weight:1000; font-size:14px; }}
        .btn:hover {{ filter: brightness(1.2); }}
        .btn:disabled {{ opacity:.4; cursor:not-allowed; filter: grayscale(0.5); }}
        .btn.success {{ background:#064e3b; border-color:#10b981; }}
        .btn.danger {{ background:#3f0a0a; border-color:#ef4444; }}
        .btn.warning {{ background:#3b2a06; border-color:#f59e0b; }}
        .btn.info {{ background:#072a3f; border-color:#38bdf8; }}
        .btn.ghost {{ background:transparent; border-color:#2b3a5a; }}
        .topbar {{ display:flex; align-items:center; gap:12px; margin-bottom:12px; }}
        .toptitle {{ font-weight:1000; font-size:16px; }}
        .subbox {{ background:#111827; border:1px solid #26324d; border-radius:14px; padding:12px; }}
        .reqrow {{ display:flex; align-items:center; gap:8px; padding:6px 0; border-bottom: 1px solid rgba(38,50,77,.35); }}
        .chip {{ border:1px solid #2b3a5a; border-radius:14px; padding:10px 12px; }}
        .chipttl {{ font-weight:1000; font-size:13px; }}
        .chipdet {{ opacity:.85; font-size:11px; margin-top:4px; }}
        .logbox {{ height: 130px; overflow:auto; background:#0a1020; border:1px solid #26324d; border-radius:14px; padding:10px; }}
        pre {{ margin:0; white-space: pre-wrap; font-size:12px; color:#cbd5e7; }}
    </style>
    <div class="wrap">
        <script>
            function daichAction(a){{
                google.colab.kernel.invokeFunction('daich_ui_action', [a], {{}});
            }}
        </script>
        {inner}
    </div>
    """

# --------------------------------------------------------------------------------------------------
# Pantallas
# --------------------------------------------------------------------------------------------------
def _screen_html():
    scr = _UI["screen"]
    log_html = f'<div class="card"><div class="cardtitle">Panel de acciones</div><div class="logbox"><pre>{_UI["log"] or "Sin acciones recientes."}</pre></div></div>'

    if scr == "welcome":
        return _wrap(f"""<div class="card"><div class="bigtitle">🤖 DAICH — Panel de Control</div><div class="subtitle">Sistema de inspección y análisis.</div></div>
        <div class="card"><div class="cardtitle">Inicio rápido</div><div class="p">Presiona <b>¡Comencemos!</b> para operar. La interfaz validará tus archivos automáticamente.</div></div>
        <div class="center">{_btn("🚀 ¡Comencemos!", "go_home", "success")}</div>""")

    if scr == "home":
        return _wrap(f"""{_topbar("Menú principal")}
        <div class="card"><div class="bigtitle">🧭 Menú principal</div><div class="subtitle">Selecciona una operación.</div></div>
        <div class="grid1">{_btn("1️⃣ Modo detección", "go_detection", "danger")}
        {_btn("2️⃣ Modo entrenamiento 🔒", "none", "warning", disabled=True)}
        {_btn("3️⃣ Ver estado del sistema 🔒", "go_status", "info", disabled=True)}</div>""")

    if scr == "status":
        return _wrap(f'{_topbar("Estado del sistema")}{_status_html()}')

    if scr == "detection":
        req = ["zip_engine", "video"]
        pres = _presence()
        can_run_basic = (len(_missing(req)) == 0)
        can_run_tech = can_run_basic and pres["has_csv"]

        return _wrap(f"""{_topbar("Modo detección")}
        <div class="card"><div class="cardtitle">📌 Faltantes</div><div class="p"><b>{_missing_text(req)}</b></div></div>
        {_requirements_html(req, "📥 Checklist detección")}
        <div class="grid1">{_btn("📤 Subir archivos", "det_upload", "primary")}</div>
        <div class="grid1" style="margin-top:10px;">
            {_btn("▶️ detección respaldada con datos técnicos", "det_run", "success", disabled=(not can_run_tech))}
            {_btn("▶️ detección con dimensionamiento autocontenido", "det_run_auto", "info", disabled=(not can_run_basic))}
        </div>
        {log_html}""")

# --------------------------------------------------------------------------------------------------
# Lógica de Control
# --------------------------------------------------------------------------------------------------
def _render():
    clear_output(wait=True); display(HTML(_screen_html()))

def _run_motor(title, fn):
    buf = io.StringIO()
    try:
        with _SilenceUltralytics(), contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            print(f"⏳ Ejecutando: {title}..."); fn(); print("✅ Finalizado.")
    except Exception as e: print(f"❌ Error: {str(e)}", file=buf)
    _UI["log"] = buf.getvalue(); _render()

def daich_ui_action(action):
    if action == "go_home": _UI["log"] = ""; _UI["nav"] = ["welcome", "home"]; _UI["screen"] = "home"
    elif action == "go_status": _UI["screen"] = "status"; _UI["nav"].append("status")
    elif action == "back":
        if len(_UI["nav"]) > 1: _UI["nav"].pop(); _UI["screen"] = _UI["nav"][-1]

    elif action == "go_detection": _UI["screen"] = "detection"; _UI["nav"].append("detection")
    elif action == "det_upload": _run_motor("Carga de archivos", subida_archivos)

    elif action == "det_run":
        _UI["log"] = "⏳ Procesando reporte técnico..."; _render()
        _run_motor("Detección y Reporte", detectar)

    elif action == "det_run_auto":
        _UI["log"] = "⏳ Procesando autodimensionamiento..."; _render()
        _run_motor("Autodimensionamiento", autodimension)

    _render()

output.register_callback("daich_ui_action", daich_ui_action)
_render()